## 2.1 — What is Pandas?

**Pandas** is the tool every data engineer uses to work with **tables of data**.

> 🗂️ Think of it as a super-powered Excel spreadsheet you can program.

| Structure | Looks Like | Think of It As |
|-----------|------------|----------------|
| **Series** | A single column | One column in Excel |
| **DataFrame** | A full table | A complete Excel sheet |

Our World Cup dataset has **852 rows** and **20 columns** — perfect for Pandas!

In [8]:
import pandas as pd
import numpy as np

# ── Load and clean the dataset ──
df = pd.read_csv("WorldCupMatches.csv")

# Drop empty rows (the CSV has blank rows between tournaments)
df = df.dropna(subset=["Year", "Home Team Name", "Home Team Goals", "Away Team Goals"])

# Fix data types
df["Year"]            = df["Year"].astype(int)
df["Home Team Goals"] = df["Home Team Goals"].astype(int)
df["Away Team Goals"] = df["Away Team Goals"].astype(int)
df["Attendance"]      = pd.to_numeric(df["Attendance"], errors="coerce")

# Strip extra spaces from string columns
df["Home Team Name"] = df["Home Team Name"].str.strip()
df["Away Team Name"] = df["Away Team Name"].str.strip()
df["City"]           = df["City"].str.strip()
df["Stadium"]        = df["Stadium"].str.strip()

print(f"✔ Dataset ready: {df.shape[0]} matches, {df.shape[1]} columns")
print(f"World Cups covered: {df['Year'].min()} → {df['Year'].max()}")

✔ Dataset ready: 852 matches, 20 columns
World Cups covered: 1930 → 2014


## 2.2 — First Look at the Data

In [9]:
# ── Always start by looking at the data ──
print("=== FIRST 5 ROWS ===")
print(df.head())

=== FIRST 5 ROWS ===
   Year              Datetime    Stage         Stadium        City  \
0  1930  13 Jul 1930 - 15:00   Group 1         Pocitos  Montevideo   
1  1930  13 Jul 1930 - 15:00   Group 4  Parque Central  Montevideo   
2  1930  14 Jul 1930 - 12:45   Group 2  Parque Central  Montevideo   
3  1930  14 Jul 1930 - 14:50   Group 3         Pocitos  Montevideo   
4  1930  15 Jul 1930 - 16:00   Group 1  Parque Central  Montevideo   

  Home Team Name  Home Team Goals  Away Team Goals Away Team Name  \
0         France                4                1         Mexico   
1            USA                3                0        Belgium   
2     Yugoslavia                2                1         Brazil   
3        Romania                3                1           Peru   
4      Argentina                1                0         France   

  Win conditions  Attendance  Half-time Home Goals  Half-time Away Goals  \
0                     4444.0                   3.0                 

In [10]:
print("=== COLUMN NAMES & DATA TYPES ===")
print(df.dtypes)
print()
print("=== QUICK STATISTICS ===")
print(df[["Home Team Goals","Away Team Goals","Attendance"]].describe().round(2))

=== COLUMN NAMES & DATA TYPES ===
Year                      int64
Datetime                    str
Stage                       str
Stadium                     str
City                        str
Home Team Name              str
Home Team Goals           int64
Away Team Goals           int64
Away Team Name              str
Win conditions              str
Attendance              float64
Half-time Home Goals    float64
Half-time Away Goals    float64
Referee                     str
Assistant 1                 str
Assistant 2                 str
RoundID                 float64
MatchID                 float64
Home Team Initials          str
Away Team Initials          str
dtype: object

=== QUICK STATISTICS ===
       Home Team Goals  Away Team Goals  Attendance
count           852.00           852.00      850.00
mean              1.81             1.02    45164.80
std               1.61             1.09    23485.25
min               0.00             0.00     2000.00
25%               1.00    

In [11]:
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
print(missing[missing > 0])     # only show columns that HAVE missing values

=== MISSING VALUES ===
Attendance    2
dtype: int64


## 2.3 — Adding New Columns (Feature Engineering)

Data engineers often **create new columns** from existing ones to make analysis easier.

In [12]:
# ── Add useful calculated columns ──
df["Total Goals"]    = df["Home Team Goals"] + df["Away Team Goals"]
df["Goal Diff"]      = df["Home Team Goals"] - df["Away Team Goals"]  # positive = home won

# ── Result for the home team ──
df["Home Result"] = df["Goal Diff"].apply(
    lambda x: "Win" if x > 0 else ("Draw" if x == 0 else "Loss")
)

# ── High-scoring match flag ──
df["Thriller"] = df["Total Goals"] >= 5    # True/False column

print("Columns after adding new ones:")
print(df.columns.tolist())
print()
print("Sample rows with new columns:")
print(df[["Year","Home Team Name","Home Team Goals","Away Team Goals",
          "Total Goals","Home Result","Thriller"]].head(8))

Columns after adding new ones:
['Year', 'Datetime', 'Stage', 'Stadium', 'City', 'Home Team Name', 'Home Team Goals', 'Away Team Goals', 'Away Team Name', 'Win conditions', 'Attendance', 'Half-time Home Goals', 'Half-time Away Goals', 'Referee', 'Assistant 1', 'Assistant 2', 'RoundID', 'MatchID', 'Home Team Initials', 'Away Team Initials', 'Total Goals', 'Goal Diff', 'Home Result', 'Thriller']

Sample rows with new columns:
   Year Home Team Name  Home Team Goals  Away Team Goals  Total Goals  \
0  1930         France                4                1            5   
1  1930            USA                3                0            3   
2  1930     Yugoslavia                2                1            3   
3  1930        Romania                3                1            4   
4  1930      Argentina                1                0            1   
5  1930          Chile                3                0            3   
6  1930     Yugoslavia                4                0      

## 2.4 — Selecting & Filtering

In [13]:
# ── Select specific columns ──
print("=== MATCH SCORELINES ===")
scorelines = df[["Year","Home Team Name","Home Team Goals","Away Team Goals","Away Team Name"]]
print(scorelines.head(6))

=== MATCH SCORELINES ===
   Year Home Team Name  Home Team Goals  Away Team Goals Away Team Name
0  1930         France                4                1         Mexico
1  1930            USA                3                0        Belgium
2  1930     Yugoslavia                2                1         Brazil
3  1930        Romania                3                1           Peru
4  1930      Argentina                1                0         France
5  1930          Chile                3                0         Mexico


In [14]:
# ── Filter: only Final matches ──
finals = df[df["Stage"] == "Final"]
print("=== ALL WORLD CUP FINALS ===")
print(finals[["Year","Home Team Name","Home Team Goals",
               "Away Team Goals","Away Team Name"]].to_string(index=False))

=== ALL WORLD CUP FINALS ===
 Year Home Team Name  Home Team Goals  Away Team Goals Away Team Name
 1930        Uruguay                4                2      Argentina
 1934          Italy                2                1 Czechoslovakia
 1938          Italy                4                2        Hungary
 1954     Germany FR                3                2        Hungary
 1958         Brazil                5                2         Sweden
 1962         Brazil                3                1 Czechoslovakia
 1966        England                4                2     Germany FR
 1970         Brazil                4                1          Italy
 1974    Netherlands                1                2     Germany FR
 1978      Argentina                3                1    Netherlands
 1982          Italy                3                1     Germany FR
 1986      Argentina                3                2     Germany FR
 1990     Germany FR                1                0      A

In [15]:
# ── Filter: high-scoring thrillers ──
thrillers = df[df["Total Goals"] >= 7].sort_values("Total Goals", ascending=False)
print("=== HIGHEST SCORING MATCHES EVER (7+ goals) ===")
print(thrillers[["Year","Home Team Name","Home Team Goals",
                  "Away Team Goals","Away Team Name","Total Goals"]]
      .to_string(index=False))

=== HIGHEST SCORING MATCHES EVER (7+ goals) ===
 Year Home Team Name  Home Team Goals  Away Team Goals Away Team Name  Total Goals
 1954        Austria                7                5    Switzerland           12
 1954        Hungary                8                3     Germany FR           11
 1938         Brazil                6                5         Poland           11
 1982        Hungary               10                1    El Salvador           11
 1958         France                7                3       Paraguay           10
 1954        Hungary                9                0 Korea Republic            9
 1958         France                6                3     Germany FR            9
 1954     Germany FR                7                2         Turkey            9
 1974     Yugoslavia                9                0          Zaire            9
 1930      Argentina                6                3         Mexico            9
 1934          Italy                7  

In [16]:
# ── Filter: Pakistan — oh wait, Pakistan has never qualified!
# Let's check a team we know — Brazil ──
brazil = df[(df["Home Team Name"] == "Brazil") | (df["Away Team Name"] == "Brazil")]
print(f"Brazil played {len(brazil)} World Cup matches")
print()
brazil_wins  = df[(df["Home Team Name"]  == "Brazil") & (df["Home Result"] == "Win")]
brazil_wins2 = df[(df["Away Team Name"]  == "Brazil") & (df["Home Result"] == "Loss")]
total_wins   = len(brazil_wins) + len(brazil_wins2)
print(f"Brazil won {total_wins} of those {len(brazil)} matches")
print(f"Brazil win rate: {total_wins/len(brazil)*100:.1f}%")

Brazil played 108 World Cup matches

Brazil won 71 of those 108 matches
Brazil win rate: 65.7%


## 2.5 — Sorting & Ranking

In [17]:
# ── Sort to find the most goals ever scored in a match ──
print("=== TOP 10 HIGHEST SCORING MATCHES ===")
top_matches = df.sort_values("Total Goals", ascending=False).head(10)
print(top_matches[["Year","Home Team Name","Home Team Goals",
                    "Away Team Goals","Away Team Name","Total Goals"]]
      .to_string(index=False))

=== TOP 10 HIGHEST SCORING MATCHES ===
 Year Home Team Name  Home Team Goals  Away Team Goals Away Team Name  Total Goals
 1954        Austria                7                5    Switzerland           12
 1938         Brazil                6                5         Poland           11
 1982        Hungary               10                1    El Salvador           11
 1954        Hungary                8                3     Germany FR           11
 1958         France                7                3       Paraguay           10
 1954        Hungary                9                0 Korea Republic            9
 1954     Germany FR                7                2         Turkey            9
 1974     Yugoslavia                9                0          Zaire            9
 1958         France                6                3     Germany FR            9
 1930      Argentina                6                3         Mexico            9


## 2.6 — groupby() — Summarising by Category

`groupby()` is one of the most powerful Pandas tools. It answers questions like:
*'What is the average goals per match in each World Cup year?'*

In [18]:
# ── Total goals and matches per World Cup ──
yearly = df.groupby("Year").agg(
    Matches      = ("Total Goals", "count"),
    Total_Goals  = ("Total Goals", "sum"),
    Avg_Goals    = ("Total Goals", "mean"),
    Avg_Crowd    = ("Attendance",  "mean"),
).round(2)

print("=== WORLD CUP STATISTICS BY YEAR ===")
print(yearly.to_string())

=== WORLD CUP STATISTICS BY YEAR ===
      Matches  Total_Goals  Avg_Goals  Avg_Crowd
Year                                            
1930       18           70       3.89   32808.28
1934       17           70       4.12   21352.94
1938       18           84       4.67   20872.22
1950       22           88       4.00   47511.18
1954       26          140       5.38   29561.81
1958       35          126       3.60   23423.14
1962       32           89       2.78   27911.62
1966       32           89       2.78   48847.97
1970       32           95       2.97   50124.22
1974       38           97       2.55   49098.76
1978       38          102       2.68   40678.71
1982       52          146       2.81   40571.60
1986       52          132       2.54   46039.06
1990       52          115       2.21   48388.75
1994       52          141       2.71   68991.12
1998       64          171       2.67   43517.19
2002       64          161       2.52   42268.70
2006       64          147      

In [19]:
# ── Top 15 teams by total goals scored ──
# A team can be home OR away, so we need to count both
home_scored = df.groupby("Home Team Name")["Home Team Goals"].sum()
away_scored = df.groupby("Away Team Name")["Away Team Goals"].sum()

# Combine them (some team names only appear in one column)
total_scored = home_scored.add(away_scored, fill_value=0).sort_values(ascending=False)

print("=== TOP 15 GOAL-SCORING NATIONS (all time) ===")
for i, (team, goals) in enumerate(total_scored.head(15).items(), 1):
    bar = "█" * int(goals // 10)
    print(f"  {i:>2}. {team:<25} {int(goals):>4} goals  {bar}")

=== TOP 15 GOAL-SCORING NATIONS (all time) ===
   1. Brazil                     225 goals  ██████████████████████
   2. Argentina                  133 goals  █████████████
   3. Germany FR                 131 goals  █████████████
   4. Italy                      128 goals  ████████████
   5. France                     108 goals  ██████████
   6. Germany                    104 goals  ██████████
   7. Spain                       92 goals  █████████
   8. Netherlands                 91 goals  █████████
   9. Hungary                     87 goals  ████████
  10. Uruguay                     80 goals  ████████
  11. England                     79 goals  ███████
  12. Sweden                      74 goals  ███████
  13. Yugoslavia                  60 goals  ██████
  14. Mexico                      58 goals  █████
  15. Belgium                     54 goals  █████


In [20]:
# ── Most appearances (matches played) by each team ──
home_apps = df["Home Team Name"].value_counts()
away_apps = df["Away Team Name"].value_counts()
total_apps = home_apps.add(away_apps, fill_value=0).sort_values(ascending=False)

print("=== TOP 10 TEAMS BY WORLD CUP APPEARANCES ===")
for i, (team, apps) in enumerate(total_apps.head(10).items(), 1):
    print(f"  {i:>2}. {team:<25} {int(apps):>4} matches")

=== TOP 10 TEAMS BY WORLD CUP APPEARANCES ===
   1. Brazil                     108 matches
   2. Italy                       83 matches
   3. Argentina                   81 matches
   4. England                     62 matches
   5. Germany FR                  62 matches
   6. France                      61 matches
   7. Spain                       59 matches
   8. Mexico                      54 matches
   9. Netherlands                 54 matches
  10. Uruguay                     52 matches


## 2.7 — Handling Missing Data

Real-world data is always messy — missing values are normal. A good data engineer knows how to find and fix them.

In [21]:
# ── Check Attendance column ──
print(f"Total matches     : {len(df)}")
print(f"Attendance missing: {df['Attendance'].isnull().sum()}")
print()

# ── Fill missing attendance with the year's average ──
df["Attendance"] = df.groupby("Year")["Attendance"].transform(
    lambda x: x.fillna(x.mean())
)

print(f"Missing after fix : {df['Attendance'].isnull().sum()}")
print()
print("Attendance stats after cleaning:")
print(df["Attendance"].describe().round(0))

Total matches     : 852
Attendance missing: 2

Missing after fix : 0

Attendance stats after cleaning:
count       852.0
mean      45189.0
std       23463.0
min        2000.0
25%       30000.0
50%       41788.0
75%       61362.0
max      173850.0
Name: Attendance, dtype: float64


---
## 🎯 10-MINUTE ACTIVITY — Hour 2
### *Build a World Cup Team Report Card*

Choose your favourite team and build a complete stats profile!
The code works for ANY team in the dataset.

---

In [22]:
# 🎯 ACTIVITY 2 — Team Report Card
# Change this to any team you like!
TEAM = "Germany FR"

# ── Get all matches involving this team ──
team_matches = df[(df["Home Team Name"] == TEAM) | (df["Away Team Name"] == TEAM)].copy()

# ── For each match, figure out if the team won/drew/lost ──
results = []
for _, row in team_matches.iterrows():
    if row["Home Team Name"] == TEAM:
        gf = row["Home Team Goals"]    # goals FOR
        ga = row["Away Team Goals"]    # goals AGAINST
    else:
        gf = row["Away Team Goals"]
        ga = row["Home Team Goals"]
    if gf > ga:   results.append("Win")
    elif gf == ga: results.append("Draw")
    else:          results.append("Loss")

team_matches["Result"] = results
team_matches["GF"]     = team_matches.apply(
    lambda r: r["Home Team Goals"] if r["Home Team Name"] == TEAM else r["Away Team Goals"], axis=1)
team_matches["GA"]     = team_matches.apply(
    lambda r: r["Away Team Goals"] if r["Home Team Name"] == TEAM else r["Home Team Goals"], axis=1)

wins   = results.count("Win")
draws  = results.count("Draw")
losses = results.count("Loss")
total  = len(results)

print("=" * 50)
print(f"  WORLD CUP REPORT CARD: {TEAM}")
print("=" * 50)
print(f"  Tournaments  : {team_matches['Year'].nunique()}")
print(f"  Matches      : {total}")
print(f"  Wins         : {wins}  ({wins/total*100:.0f}%)")
print(f"  Draws        : {draws}  ({draws/total*100:.0f}%)")
print(f"  Losses       : {losses}  ({losses/total*100:.0f}%)")
print(f"  Goals scored : {int(team_matches['GF'].sum())}")
print(f"  Goals conceded: {int(team_matches['GA'].sum())}")
print(f"  Goal difference: {int(team_matches['GF'].sum() - team_matches['GA'].sum())}")
print()
print("  Results by year:")
yr_results = team_matches.groupby("Year")["Result"].value_counts().unstack(fill_value=0)
print(yr_results.to_string())

  WORLD CUP REPORT CARD: Germany FR
  Tournaments  : 10
  Matches      : 62
  Wins         : 36  (58%)
  Draws        : 14  (23%)
  Losses       : 12  (19%)
  Goals scored : 131
  Goals conceded: 77
  Goal difference: 54

  Results by year:
Result  Draw  Loss  Win
Year                   
1954       0     1    5
1958       2     2    2
1962       1     1    2
1966       1     1    4
1970       0     1    5
1974       0     1    6
1978       4     1    1
1982       2     2    3
1986       2     2    3
1990       2     0    5


In [23]:
# 🔥 Bonus: Which team has the WORST World Cup record? (most losses)
all_teams = set(df["Home Team Name"].unique()) | set(df["Away Team Name"].unique())

worst = []
for team in all_teams:
    matches = df[(df["Home Team Name"] == team) | (df["Away Team Name"] == team)]
    losses  = sum(
        1 for _, r in matches.iterrows()
        if (r["Home Team Name"] == team and r["Home Result"] == "Loss") or
           (r["Away Team Name"] == team and r["Home Result"] == "Win")
    )
    worst.append((team, losses, len(matches)))

worst_df = pd.DataFrame(worst, columns=["Team","Losses","Matches"])
worst_df["Loss Rate"] = (worst_df["Losses"] / worst_df["Matches"] * 100).round(1)
worst_df = worst_df[worst_df["Matches"] >= 10].sort_values("Loss Rate", ascending=False)
print("Teams with the worst World Cup loss rate (min 10 matches):")
print(worst_df.head(8).to_string(index=False))

Teams with the worst World Cup loss rate (min 10 matches):
          Team  Losses  Matches  Loss Rate
  Saudi Arabia       9       13       69.2
     Australia       8       13       61.5
           USA      20       34       58.8
       Tunisia       7       12       58.3
       Nigeria      11       19       57.9
      Bulgaria      15       26       57.7
       Algeria       8       14       57.1
Korea Republic      17       31       54.8


In [24]:
df.to_csv("cleaned_data.csv", header=True)